<a href="https://colab.research.google.com/github/JeysonCarmona/PPMI_INVESTIGATION/blob/main/Notebook6_Patient_File_Extraction_Manager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 6 — Patient File Extraction Manager

**Objective:** Given a `PATNO`, display all already indexed information (clinical, images, exome) and allow **selective and manual** extraction: all patient images, a specific image, or the complete exome — with direct download option to the computer.

This notebook reuses exactly the same mechanism already tested in Notebook 4 (`unrar lb -r` to list, `unrar x` to extract specifically, streaming reading of `.tar.gz`). It does not re-index the entire RAR if the `rutas_dicom_completas.csv` cache already exists, does not rebuild `paciente_master_index.csv`, and does not modify any existing index.

In [6]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [7]:
!apt-get install -y unrar -qq


## 0. Path Configuration

Adjust these paths if they differ in your Drive (they are the same as used by Notebook 4).

In [8]:
import os

BASE_DIR = "/content/drive/MyDrive/Investigación_Parkinson"

# ADJUST if your actual structure is different
RAR_IMAGENES = BASE_DIR + "/Imagenes/Imagenes_PPMI.rar"
DIR_EXOMAS = BASE_DIR + "/Exomas - Parkinson"
RESULTADOS_DIR = BASE_DIR + "/Jeyson_Carmona_Michael_Lamprea/segunda entrega/Resultados"

MASTER_INDEX = RESULTADOS_DIR + "/paciente_master_index.csv"
RUTA_EXOMAS_INDEX = RESULTADOS_DIR + "/exomas_index.csv"
RUTA_CACHE_RUTAS_DICOM = RESULTADOS_DIR + "/rutas_dicom_completas.csv"

# Output folder in Drive for extracted files (persists between sessions)
EXTRACCIONES_DIR = RESULTADOS_DIR + "/Extracciones"
os.makedirs(EXTRACCIONES_DIR, exist_ok=True)

for nombre, ruta in [("Master Index", MASTER_INDEX), ("Exome Index", RUTA_EXOMAS_INDEX)]:
    assert os.path.exists(ruta), f"'{nombre}' not found: {ruta}"
    print(f"{nombre} OK: {ruta}")

for nombre, ruta in [("RAR File", RAR_IMAGENES), ("TAR Folder", DIR_EXOMAS)]:
    estado = "OK" if os.path.exists(ruta) else "NOT FOUND (only needed for extraction)"
    print(f"{nombre}: {ruta} -> {estado}")

print("rutas_dicom_completas.csv cache:", "exists" if os.path.exists(RUTA_CACHE_RUTAS_DICOM) else "does not exist yet (will be built in Section 2)")

Master Index OK: /content/drive/MyDrive/Investigación_Parkinson/Jeyson_Carmona_Michael_Lamprea/segunda entrega/Resultados/paciente_master_index.csv
Exome Index OK: /content/drive/MyDrive/Investigación_Parkinson/Jeyson_Carmona_Michael_Lamprea/segunda entrega/Resultados/exomas_index.csv
RAR File: /content/drive/MyDrive/Investigación_Parkinson/Imagenes/Imagenes_PPMI.rar -> OK
TAR Folder: /content/drive/MyDrive/Investigación_Parkinson/Exomas - Parkinson -> OK
rutas_dicom_completas.csv cache: exists


## 1. Load Existing Indexes (Read-Only)

`paciente_master_index.csv` and `exomas_index.csv` are read as is, with their confirmed actual column names.

In [9]:
import pandas as pd

df_master = pd.read_csv(MASTER_INDEX)
df_master["PATNO"] = df_master["PATNO"].astype(int)

df_exomas_idx = pd.read_csv(RUTA_EXOMAS_INDEX)
df_exomas_idx["PATNO"] = df_exomas_idx["PATNO"].astype(int)

print("paciente_master_index.csv:", df_master.shape)
print("exomas_index.csv:        ", df_exomas_idx.shape)


paciente_master_index.csv: (8619, 25)
exomas_index.csv:         (645, 4)


## 2. Complete Image Paths Cache

`imagenes_index.csv` is aggregated by study (it doesn't provide individual paths), so `rutas_dicom_completas.csv` is used to display/extract specific files. If it already exists (your case), it's loaded directly. If it doesn't exist, it's built **only once** by reading only the RAR headers (`unrar lb -r`, never unzips content) — this might take several minutes the first time.

In [10]:
import subprocess
import re

def listar_rutas_rar(ruta_rar, unrar_path="unrar"):
    """Reads only the internal names of the RAR via 'unrar lb -r' (list bare,
    recursive). It only reads headers: it does not extract or decompress content."""
    comando = [unrar_path, "lb", "-r", ruta_rar]
    proceso = subprocess.Popen(
        comando, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1,
    )
    rutas_archivos = []
    for linea in proceso.stdout:
        linea = linea.rstrip("\n").replace("\\", "/")
        if linea and not linea.endswith("/"):
            rutas_archivos.append(linea)
    proceso.stdout.close()
    codigo_retorno = proceso.wait()
    stderr_out = proceso.stderr.read()
    proceso.stderr.close()
    if codigo_retorno != 0:
        raise RuntimeError(f"unrar returned code {codigo_retorno}. Details: {stderr_out}")
    return rutas_archivos


PATRON_GRUPO = re.compile(r"(Control|Parkinson_Disease|PD)", re.IGNORECASE)
PATRON_VISTA = re.compile(r"(Axial|Sagital)", re.IGNORECASE)
PATRON_SECUENCIA = re.compile(r"(T1|T2)", re.IGNORECASE)
PATRON_DIMENSION = re.compile(r"(2D|3D)", re.IGNORECASE)
PATRON_FECHA = re.compile(r"^(\d{4}-\d{2}-\d{2})")

def parsear_ruta_dicom(ruta):
    partes = ruta.split("/")
    grupo = vista = secuencia = dimension = patno = estudio_desc = fecha = None

    m = PATRON_GRUPO.search(ruta)
    if m: grupo = m.group(1)
    m = PATRON_VISTA.search(ruta)
    if m: vista = m.group(1)
    m = PATRON_SECUENCIA.search(ruta)
    if m: secuencia = m.group(1)
    m = PATRON_DIMENSION.search(ruta)
    if m: dimension = m.group(1)

    for i, parte in enumerate(partes):
        if parte.upper() == "PPMI" and i + 1 < len(partes):
            posible_patno = partes[i + 1]
            if posible_patno.isdigit():
                patno = int(posible_patno)
                if i + 2 < len(partes):
                    estudio_desc = partes[i + 2]
                if i + 3 < len(partes):
                    m_fecha = PATRON_FECHA.match(partes[i + 3])
                    if m_fecha:
                        fecha = m_fecha.group(1)
            break

    return {
        "ruta_completa": ruta, "grupo_diagnostico": grupo, "vista": vista,
        "secuencia": secuencia, "dimension": dimension, "PATNO": patno,
        "estudio": estudio_desc, "fecha": fecha,
        "nombre_archivo": partes[-1] if partes else None,
    }


if os.path.exists(RUTA_CACHE_RUTAS_DICOM):
    df_rutas_dicom = pd.read_csv(RUTA_CACHE_RUTAS_DICOM)
    print(f"Cache loaded from disk: {len(df_rutas_dicom):,} paths.")
else:
    print("Cache does not exist yet. Scanning RAR headers (this may take a while)...")
    rutas = listar_rutas_rar(RAR_IMAGENES)
    print(f"Paths read: {len(rutas):,}. Parsing...")
    registros = [parsear_ruta_dicom(r) for r in rutas]
    df_rutas_dicom = pd.DataFrame(registros)
    df_rutas_dicom.to_csv(RUTA_CACHE_RUTAS_DICOM, index=False)
    print(f"Cache saved to: {RUTA_CACHE_RUTAS_DICOM}")

df_rutas_dicom["PATNO"] = pd.to_numeric(df_rutas_dicom["PATNO"], errors="coerce")

/tmp/ipykernel_1194/3059000321.py:66: DtypeWarning: Columns (3,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_rutas_dicom = pd.read_csv(RUTA_CACHE_RUTAS_DICOM)


Cache loaded from disk: 3,277,408 paths.


## 3. Load Patient

Enter the `PATNO` (for example, the one you selected in Notebook 5) and click **Load patient**.

In [11]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

def valor_o_na(fila, col):
    if col not in fila.index or pd.isna(fila[col]):
        return "not available"
    valor = fila[col]
    if isinstance(valor, float) and valor.is_integer():
        return int(valor)
    return valor


estado_paciente_actual = {
    "patno": None,
    "fila_master": None,
    "df_imagenes_paciente": None,   # all individual paths, from cache
    "info_exoma": None,             # dict with archivo_tar / archivo_vcf
}


def localizar_exoma(patno):
    fila = df_exomas_idx[df_exomas_idx["PATNO"] == patno]
    if fila.empty:
        return None
    fila = fila.iloc[0]
    return {
        "PATNO": patno,
        "archivo_tar": fila["archivo_tar"],
        "archivo_vcf": fila["archivo_vcf"],
    }


def cargar_paciente(patno_texto):
    patno_texto = str(patno_texto).strip()
    if not patno_texto.isdigit():
        return False, "Enter a valid numeric PATNO."
    patno = int(patno_texto)

    filas_master = df_master[df_master["PATNO"] == patno]
    if filas_master.empty:
        return False, f"PATNO {patno} not found in paciente_master_index.csv."

    estado_paciente_actual["patno"] = patno
    estado_paciente_actual["fila_master"] = filas_master.iloc[0]
    estado_paciente_actual["df_imagenes_paciente"] = df_rutas_dicom[df_rutas_dicom["PATNO"] == patno].copy()
    estado_paciente_actual["info_exoma"] = localizar_exoma(patno)

    n_img = len(estado_paciente_actual["df_imagenes_paciente"])
    tiene_exo = estado_paciente_actual["info_exoma"] is not None
    texto_exoma = "found" if tiene_exo else "not found"
    return True, f"Patient {patno} loaded: {n_img} image paths found, exome {texto_exoma}."


In [19]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output


nb6_patno_input = widgets.Text(placeholder="Ej. 3184", description="PATNO:", layout=widgets.Layout(width="250px"))
nb6_boton_cargar = widgets.Button(description="Load patient", button_style="primary", icon="upload")
nb6_estado_carga = widgets.HTML(value="<i>No patient loaded yet.</i>")

nb6_resumen_out = widgets.Output()
nb6_rutas_img_out = widgets.Output()
nb6_ruta_exo_out = widgets.Output()
nb6_extraccion_out = widgets.Output()
nb6_export_out = widgets.Output()


def nb6_render_resumen():
    with nb6_resumen_out:
        clear_output()
        fila = estado_paciente_actual["fila_master"]
        if fila is None:
            return
        n_img_reales = len(estado_paciente_actual["df_imagenes_paciente"])
        exo = estado_paciente_actual["info_exoma"]

        html = f"""
        <div style=\"border:1px solid #ccc; border-radius:8px; padding:16px; font-family:sans-serif; max-width:700px;\">
          <h3 style=\"margin-top:0;\">Patient {valor_o_na(fila, 'PATNO')}</h3>

          <h4 style=\"color:#2c5282;\">Clinical Information</h4>
          <ul>
            <li><b>COHORT:</b> {valor_o_na(fila, 'COHORT')}</li>
            <li><b>COHORT_DEFINITION:</b> {valor_o_na(fila, 'COHORT_DEFINITION')}</li>
            <li><b>ENROLL_STATUS:</b> {valor_o_na(fila, 'ENROLL_STATUS')}</li>
            <li><b>ENROLL_DATE:</b> {valor_o_na(fila, 'ENROLL_DATE')}</li>
            <li><b>ENROLL_AGE:</b> {valor_o_na(fila, 'ENROLL_AGE')}</li>
          </ul>

          <h4 style=\"color:#2f855a;\">Image Information</h4>
          <ul>
            <li><b>Has images:</b> {"Yes" if bool(fila.get('Tiene_imagenes')) else "No"}</li>
            <li><b>N_total_studies (master index):</b> {valor_o_na(fila, 'N_estudios_total')}</li>
            <li><b>N_total_series (master index):</b> {valor_o_na(fila, 'N_series_total')}</li>
            <li><b>N_total_dicom_files (master index):</b> {valor_o_na(fila, 'N_archivos_dicom_total')}</li>
            <li><b>DICOM files currently located in RAR:</b> {n_img_reales}</li>
            <li><b>Diagnostic group (images):</b> {valor_o_na(fila, 'Grupo_diagnostico_imagenes')}</li>
          </ul>

          <h4 style=\"color:#975a16;\">Genetic Information</h4>
          <ul>
            <li><b>Has exome:</b> {"Yes" if exo is not None else "No"}</li>
            <li><b>TAR file:</b> {exo['archivo_tar'] if exo else 'not available'}</li>
            <li><b>VCF file:</b> {exo['archivo_vcf'] if exo else 'not available'}</li>
          </ul>
        </div>
        """
        display(HTML(html))


def nb6_render_rutas_imagenes():
    with nb6_rutas_img_out:
        clear_output()
        df_img = estado_paciente_actual["df_imagenes_paciente"]
        if df_img is None or df_img.empty:
            print("No indexed image paths for this patient in the cache.")
            return
        print(f"{len(df_img)} DICOM file(s) found:")
        display(df_img[["ruta_completa", "grupo_diagnostico", "vista", "secuencia", "dimension", "estudio", "fecha", "nombre_archivo"]].reset_index(drop=True))


def nb6_render_ruta_exoma():
    with nb6_ruta_exo_out:
        clear_output()
        exo = estado_paciente_actual["info_exoma"]
        if exo is None:
            print("No exome indexed for this patient.")
            return
        display(pd.DataFrame([exo]))


def nb6_al_cargar(_=None):
    ok, mensaje = cargar_paciente(nb6_patno_input.value)
    color = "#2f855a" if ok else "#c53030"
    nb6_estado_carga.value = f"<span style='color:{color}'>{mensaje}</span>"
    for out in [nb6_extraccion_out, nb6_export_out]:
        with out:
            clear_output()
    if ok:
        nb6_render_resumen()
        nb6_render_rutas_imagenes()
        nb6_render_ruta_exoma()
        nb6_actualizar_selector_dicom()
    else:
        for out in [nb6_resumen_out, nb6_rutas_img_out, nb6_ruta_exo_out]:
            with out:
                clear_output()


nb6_boton_cargar.on_click(nb6_al_cargar)
display(widgets.HBox([nb6_patno_input, nb6_boton_cargar]), nb6_estado_carga)

HTML(value='<i>No patient loaded yet.</i>')

## 4. Patient Summary

In [13]:
display(nb6_resumen_out)


Output()

## 5. All Available Paths

### Images (inside the `.rar`)

In [14]:
display(nb6_rutas_img_out)


Output()

### Exome (inside the `.tar.gz`)

In [15]:
display(nb6_ruta_exo_out)


Output()

This cell displays the `nb6_ruta_exo_out` widget. Once a patient is loaded, this widget will display a DataFrame with the exome file paths (TAR and VCF) if an exome is found for the patient.

## 6. Selective Extraction

**Nothing is extracted automatically.** Each button uses `unrar x` (or streaming reading of the TAR) pointing only to the loaded patient's paths — the complete `.rar` or `.tar.gz` is never decompressed.

Everything is first saved to Drive (`Results/Extractions/Patient_<PATNO>/...`) and then offered for direct download to the computer.

In [16]:
import tarfile

def carpeta_paciente(patno):
    carpeta = f"{EXTRACCIONES_DIR}/Paciente_{patno}"
    os.makedirs(carpeta + "/Imagenes", exist_ok=True)
    os.makedirs(carpeta + "/Exoma", exist_ok=True)
    return carpeta


def extraer_rutas_rar(rutas, destino, log, tam_lote=150):
    """Extracts only the indicated paths from the RAR, using 'unrar x' (with -ep
    to avoid replicating the internal folder tree). It groups into batches to
    avoid excessively long commands."""
    if not rutas:
        log.append("No paths to extract.")
        return []
    if not os.path.exists(RAR_IMAGENES):
        log.append(f"RAR file not found in: {RAR_IMAGENES}")
        return []

    extraidos = []
    for i in range(0, len(rutas), tam_lote):
        lote = rutas[i:i + tam_lote]
        comando = ["unrar", "x", "-y", "-ep", RAR_IMAGENES, *lote, destino + "/"]
        resultado = subprocess.run(comando, capture_output=True, text=True)
        if resultado.returncode != 0:
            log.append(f"WARNING in batch {i // tam_lote + 1}: {resultado.stderr.strip()[:300]}")
        for ruta in lote:
            nombre = ruta.split("/")[-1]
            ruta_local = os.path.join(destino, nombre)
            if os.path.exists(ruta_local):
                extraidos.append(ruta_local)
    log.append(f"OK: {len(extraidos)} of {len(rutas)} file(s) extracted in: {destino}")
    return extraidos


def extraer_vcf_paciente(archivo_tar, archivo_vcf, destino, log):
    """Opens ONLY the corresponding .tar.gz and extracts ONLY that .vcf, stopping
    as soon as it finds it (does not read the rest of the TAR)."""
    ruta_tar = os.path.join(DIR_EXOMAS, str(archivo_tar))
    if not os.path.exists(ruta_tar):
        log.append(f"TAR not found in: {ruta_tar}")
        return None

    ruta_local_vcf = os.path.join(destino, str(archivo_vcf))
    if os.path.exists(ruta_local_vcf):
        log.append(f"VCF already extracted previously: {ruta_local_vcf}")
        return ruta_local_vcf

    try:
        with tarfile.open(ruta_tar, mode="r:gz") as tar:
            miembro = None
            for m in tar:
                if m.name.endswith(str(archivo_vcf)):
                    miembro = m
                    break
            if miembro is None:
                log.append(f"'{archivo_vcf}' not found inside '{archivo_tar}'.")
                return None
            with tar.extractfile(miembro) as origen, open(ruta_local_vcf, "wb") as salida:
                while True:
                    bloque = origen.read(1024 * 1024)
                    if not bloque:
                        break
                    salida.write(bloque)
        log.append(f"OK: VCF extracted to: {ruta_local_vcf}")
        return ruta_local_vcf
    except Exception as e:
        log.append(f"ERROR extracting from TAR: {e}")
        return None

This cell defines the main functions for selective file extraction:
*   `carpeta_paciente`: Creates a dedicated folder structure in Drive for a given patient's extractions.
*   `extraer_rutas_rar`: Uses `unrar x` to extract specific files from the RAR archive to a destination directory. It processes paths in batches to avoid command line length limits.
*   `extraer_vcf_paciente`: Opens ONLY the corresponding `.tar.gz` (containing exome data) and extracts ONLY that specific VCF file, reading in streaming mode for efficiency and stopping once the VCF is found.

In [20]:
import ipywidgets as widgets
from google.colab import files as colab_files
import shutil

nb6_boton_extraer_imagenes = widgets.Button(description="Extract all images", button_style="success", icon="image", layout=widgets.Layout(width="260px"))
nb6_boton_descargar_imagenes = widgets.Button(description="Download images (.zip)", icon="download", layout=widgets.Layout(width="220px"))
nb6_boton_extraer_exoma = widgets.Button(description="Extract exome", button_style="warning", icon="flask", layout=widgets.Layout(width="180px"))
nb6_boton_descargar_exoma = widgets.Button(description="Download exome (.vcf)", icon="download", layout=widgets.Layout(width="200px"))

nb6_selector_dicom = widgets.Dropdown(options=[], description="DICOM:", layout=widgets.Layout(width="650px"))
nb6_boton_extraer_dicom = widgets.Button(description="Extract and download selected", icon="file", layout=widgets.Layout(width="280px"))

nb6_ultimo_zip_imagenes = {"ruta": None}
nb6_ultimo_vcf = {"ruta": None}


def nb6_actualizar_selector_dicom():
    df_img = estado_paciente_actual["df_imagenes_paciente"]
    if df_img is None or df_img.empty:
        nb6_selector_dicom.options = []
        return
    etiquetas = [(f"{fila['nombre_archivo']}  -  {fila['estudio']}", fila["ruta_completa"]) for _, fila in df_img.iterrows()]
    nb6_selector_dicom.options = etiquetas


def nb6_al_extraer_imagenes(_=None):
    with nb6_extraccion_out:
        clear_output()
        patno = estado_paciente_actual["patno"]
        df_img = estado_paciente_actual["df_imagenes_paciente"]
        if patno is None:
            print("First load a patient.")
            return
        if df_img is None or df_img.empty:
            print("This patient has no indexed image paths.")
            return
        carpeta = carpeta_paciente(patno)
        rutas = df_img["ruta_completa"].astype(str).tolist()
        log = []
        print(f"Extracting {len(rutas)} image file(s) for patient {patno}...")
        extraer_rutas_rar(rutas, carpeta + "/Imagenes", log)
        for linea in log:
            print(linea)


def nb6_al_descargar_imagenes(_=None):
    with nb6_extraccion_out:
        patno = estado_paciente_actual["patno"]
        if patno is None:
            print("First load a patient.")
            return
        carpeta_img = carpeta_paciente(patno) + "/Imagenes"
        if not os.path.isdir(carpeta_img) or not os.listdir(carpeta_img):
            print("No images extracted yet. First use 'Extract all images'.")
            return
        ruta_zip_base = f"/content/Paciente_{patno}_Imagenes"
        ruta_zip = shutil.make_archive(ruta_zip_base, "zip", carpeta_img)
        nb6_ultimo_zip_imagenes["ruta"] = ruta_zip
        print(f"Compressed file ready: {ruta_zip}")
        colab_files.download(ruta_zip)


def nb6_al_extraer_exoma(_=None):
    with nb6_extraccion_out:
        clear_output()
        patno = estado_paciente_actual["patno"]
        exo = estado_paciente_actual["info_exoma"]
        if patno is None:
            print("First load a patient.")
            return
        if exo is None:
            print("This patient has no indexed exome.")
            return
        carpeta = carpeta_paciente(patno)
        log = []
        print(f"Extracting exome for patient {patno} from {exo['archivo_tar']}...")
        ruta_local = extraer_vcf_paciente(exo["archivo_tar"], exo["archivo_vcf"], carpeta + "/Exoma", log)
        nb6_ultimo_vcf["ruta"] = ruta_local
        for linea in log:
            print(linea)


def nb6_al_descargar_exoma(_=None):
    with nb6_extraccion_out:
        if nb6_ultimo_vcf["ruta"] is None or not os.path.exists(nb6_ultimo_vcf["ruta"]):
            print("No VCF extracted yet. First use 'Extract exome'.")
            return
        colab_files.download(nb6_ultimo_vcf["ruta"])


def nb6_al_extraer_dicom(_=None):
    with nb6_extraccion_out:
        clear_output()
        patno = estado_paciente_actual["patno"]
        if patno is None:
            print("First load a patient.")
            return
        if not nb6_selector_dicom.value:
            print("First select a DICOM file from the list.")
            return
        carpeta = carpeta_paciente(patno)
        log = []
        print(f"Extracting individual file for patient {patno}...")
        extraidos = extraer_rutas_rar([nb6_selector_dicom.value], carpeta + "/Imagenes", log)
        for linea in log:
            print(linea)
        if extraidos:
            colab_files.download(extraidos[0])


nb6_boton_extraer_imagenes.on_click(nb6_al_extraer_imagenes)
nb6_boton_descargar_imagenes.on_click(nb6_al_descargar_imagenes)
nb6_boton_extraer_exoma.on_click(nb6_al_extraer_exoma)
nb6_boton_descargar_exoma.on_click(nb6_al_descargar_exoma)
nb6_boton_extraer_dicom.on_click(nb6_al_extraer_dicom)

display(widgets.HTML("<b>Images</b>"))
display(widgets.HBox([nb6_boton_extraer_imagenes, nb6_boton_descargar_imagenes]))
display(widgets.HTML("<b>Exome</b>"))
display(widgets.HBox([nb6_boton_extraer_exoma, nb6_boton_descargar_exoma]))
display(widgets.HTML("<hr><b>Individual DICOM Extraction:</b>"))
display(widgets.HBox([nb6_selector_dicom, nb6_boton_extraer_dicom]))
display(nb6_extraccion_out)

HTML(value='<b>Images</b>')

HTML(value='<b>Exome</b>')

HTML(value='<hr><b>Individual DICOM Extraction:</b>')

Output()

This cell configures the interactive widgets to extract and download patient data:
*   Buttons to extract/download all images, the exome, or a single selected DICOM.
*   A dropdown menu (`nb6_selector_dicom`) to select individual DICOM files for extraction.

It defines several `nb6_al_` functions as event handlers for these buttons, managing the extraction process, file compression, and triggering direct downloads using `google.colab.files`. It also updates the DICOM selector options based on the currently loaded patient.

## 7. Export Summary

Generates a CSV in `Results/Extractions/Paciente_<PATNO>/resumen_paciente_<PATNO>.csv` with clinical information, all indexed image paths, the exome path, and offers direct download.

In [21]:
import ipywidgets as widgets

nb6_boton_exportar = widgets.Button(description="Export summary", button_style="info", icon="download")


def nb6_al_exportar(_=None):
    with nb6_export_out:
        clear_output()
        patno = estado_paciente_actual["patno"]
        fila_master = estado_paciente_actual["fila_master"]
        if patno is None or fila_master is None:
            print("First load a patient.")
            return

        carpeta = carpeta_paciente(patno)
        filas_resumen = []

        for etiqueta in ["PATNO", "COHORT", "COHORT_DEFINITION", "ENROLL_STATUS", "ENROLL_DATE", "ENROLL_AGE",
                         "N_estudios_total", "N_series_total", "N_archivos_dicom_total", "Grupo_diagnostico_imagenes"]:
            filas_resumen.append({"seccion": "clinical/images", "campo": etiqueta, "valor": valor_o_na(fila_master, etiqueta)})

        df_img = estado_paciente_actual["df_imagenes_paciente"]
        if df_img is not None and not df_img.empty:
            for _, fila in df_img.iterrows():
                filas_resumen.append({"seccion": "image_path", "campo": fila["nombre_archivo"], "valor": fila["ruta_completa"]})
        else:
            filas_resumen.append({"seccion": "image_path", "campo": "-", "valor": "No indexed images"})

        exo = estado_paciente_actual["info_exoma"]
        if exo is not None:
            filas_resumen.append({"seccion": "exome", "campo": "archivo_tar", "valor": exo["archivo_tar"]})
            filas_resumen.append({"seccion": "exome", "campo": "archivo_vcf", "valor": exo["archivo_vcf"]})
        else:
            filas_resumen.append({"seccion": "exome", "campo": "-", "valor": "No indexed exome"})

        df_resumen = pd.DataFrame(filas_resumen)
        salida = f"{carpeta}/resumen_paciente_{patno}.csv"
        df_resumen.to_csv(salida, index=False)
        print(f"Summary exported to: {salida}")
        display(df_resumen)
        colab_files.download(salida)


nb6_boton_exportar.on_click(nb6_al_exportar)
display(nb6_boton_exportar, nb6_export_out)

Button(button_style='info', description='Export summary', icon='download', style=ButtonStyle())

Output()